verifying data used

In [1]:
import pandas as pd

# Security base
sec_df = pd.read_csv(
    "/lakehouse/default/Files/data/processed/securities/security_processed.csv"
)

# NSE MCAP
nse_mcap = pd.read_csv(
    "/lakehouse/default/Files/data/raw/nse_mcap/mcap05062026.csv"
)

# BSE MCAP
bse_mcap = pd.read_csv(
    "/lakehouse/default/Files/data/raw/bse_mcap/bse_mcap.csv"
)

# Sector / Industry Lookup
sector_df = pd.read_csv(
    "/lakehouse/default/Files/data/processed/securities/sector_industry_lookup.csv"
)

print("security_processed :", len(sec_df))
print("nse_mcap           :", len(nse_mcap))
print("bse_mcap           :", len(bse_mcap))
print("sector_lookup      :", len(sector_df))

StatementMeta(, 7eea9210-6569-463f-b994-5c291588057f, 3, Finished, Available, Finished, False)

security_processed : 6989
nse_mcap           : 2954
bse_mcap           : 6693
sector_lookup      : 2348


In [2]:
sec_master = (
    sec_df
    .sort_values(
        ["isin", "source"]
    )
    .drop_duplicates(
        subset=["isin"],
        keep="first"
    )
    .copy()
)

print(
    "Rows:",
    len(sec_master)
)

print(
    "Unique ISIN:",
    sec_master["isin"].nunique()
)

sec_master.head()

StatementMeta(, 7eea9210-6569-463f-b994-5c291588057f, 5, Finished, Available, Finished, False)

Rows: 4963
Unique ISIN: 4963


,symbol,company_name,series,listing_date,market_lot,face_value,isin,bse_security_code,status,group,instrument,source
198,ADSDIAG,A.D.S. Diagnostics Ltd.,EQ,NaN,1,10.0,-,523031.0,Active,Y,Equity,BSE
433,ANNVRPP,ANNVRRIDHHI VENTURES LIMITED,EQ,NaN,1,10.0,IN9075K01029,890229.0,Active,XT,Equity,BSE
3373,KRISHPP,KRISHIVAL FOODS LIMITED,EQ,NaN,1,10.0,IN90GGO01013,890232.0,Active,B,Equity,BSE
654,ATLPP,Allcargo Terminals Limited,EQ,NaN,1,2.0,IN90NN701011,890228.0,Active,B,Equity,BSE
423,ANIRITPP,ANIRIT VENTURES LIMITED,EQ,NaN,1,10.0,IN9161F01019,890231.0,Active,XT,Equity,BSE


In [3]:
# clean nse mcap
nse_mcap_clean = nse_mcap[
    [
        "Symbol",
        "Close Price/Paid up value(Rs.)",
        "Market Cap(Rs.)              "
    ]
].copy()

nse_mcap_clean = nse_mcap_clean.rename(
    columns={
        "Symbol": "symbol",
        "Close Price/Paid up value(Rs.)": "cmp",
        "Market Cap(Rs.)              ": "nse_market_cap"
    }
)

print(
    "NSE MCAP Rows:",
    len(nse_mcap_clean)
)

nse_mcap_clean.head()

StatementMeta(, 7eea9210-6569-463f-b994-5c291588057f, 8, Finished, Available, Finished, False)

NSE MCAP Rows: 2954


,symbol,cmp,nse_market_cap
0,20MICRONS,195.75,6.908038e+09
1,21STCENMGM,33.00,3.466050e+08
2,360ONE,1073.90,4.359132e+11
3,3BBLACKBIO,1120.60,9.617740e+09
4,3IINFOLTD,17.05,3.540254e+09


In [4]:
# join nse mcap 
sec_master = sec_master.merge(
    nse_mcap_clean,
    on="symbol",
    how="left"
)

print("Security Master Rows:", len(sec_master))

print(
    "CMP Found:",
    sec_master["cmp"].notna().sum()
)

print(
    "NSE MCAP Found:",
    sec_master["nse_market_cap"].notna().sum()
)

sec_master.head()

StatementMeta(, 7eea9210-6569-463f-b994-5c291588057f, 30, Finished, Available, Finished, False)

Security Master Rows: 4963
CMP Found: 2347
NSE MCAP Found: 2347


,symbol,company_name,series,listing_date,market_lot,face_value,isin,bse_security_code,status,group,instrument,source,cmp,nse_market_cap
0,ADSDIAG,A.D.S. Diagnostics Ltd.,EQ,NaN,1,10.0,-,523031.0,Active,Y,Equity,BSE,NaN,NaN
1,ANNVRPP,ANNVRRIDHHI VENTURES LIMITED,EQ,NaN,1,10.0,IN9075K01029,890229.0,Active,XT,Equity,BSE,NaN,NaN
2,KRISHPP,KRISHIVAL FOODS LIMITED,EQ,NaN,1,10.0,IN90GGO01013,890232.0,Active,B,Equity,BSE,NaN,NaN
3,ATLPP,Allcargo Terminals Limited,EQ,NaN,1,2.0,IN90NN701011,890228.0,Active,B,Equity,BSE,NaN,NaN
4,ANIRITPP,ANIRIT VENTURES LIMITED,EQ,NaN,1,10.0,IN9161F01019,890231.0,Active,XT,Equity,BSE,NaN,NaN


In [5]:
bse_mcap_clean = bse_mcap[
    [
        "ISIN_NUMBER",
        "Mktcap"
    ]
].copy()

bse_mcap_clean = bse_mcap_clean.rename(
    columns={
        "ISIN_NUMBER": "isin",
        "Mktcap": "bse_market_cap"
    }
)

print(
    "Rows:",
    len(bse_mcap_clean)
)

print(
    "Unique ISIN:",
    bse_mcap_clean["isin"].nunique()
)

bse_mcap_clean.head()

StatementMeta(, 7eea9210-6569-463f-b994-5c291588057f, 33, Finished, Available, Finished, False)

Rows: 6693
Unique ISIN: 6588


,isin,bse_market_cap
0,INE117A01022,151803.74
1,INE208C01025,27062.10
2,INE984C01013,1.46
3,INE885A01032,15561.73
4,INE432A01017,264.15


In [7]:
bse_mcap_clean = (
    bse_mcap_clean
    .drop_duplicates(
        subset=["isin"],
        keep="first"
    )
)

print("Rows:", len(bse_mcap_clean))
print("Unique ISIN:", bse_mcap_clean["isin"].nunique())

StatementMeta(, 7eea9210-6569-463f-b994-5c291588057f, 36, Finished, Available, Finished, False)

Rows: 6589
Unique ISIN: 6588


In [10]:
bse_mcap_clean = (
    bse_mcap_clean
    .dropna(
        subset=["isin"]
    )
)

print("Rows:", len(bse_mcap_clean))
print(
    "Unique ISIN:",
    bse_mcap_clean["isin"].nunique()
)

StatementMeta(, 7eea9210-6569-463f-b994-5c291588057f, 39, Finished, Available, Finished, False)

Rows: 6588
Unique ISIN: 6588


In [11]:
# Join BSE MCAP

sec_master = sec_master.merge(
    bse_mcap_clean,
    on="isin",
    how="left"
)

print(
    "BSE MCAP Found:",
    sec_master["bse_market_cap"].notna().sum()
)

# Final Market Cap

sec_master["market_cap"] = (
    sec_master["nse_market_cap"]
    .fillna(
        sec_master["bse_market_cap"]
    )
)

print(
    "Final Market Cap Found:",
    sec_master["market_cap"].notna().sum()
)

StatementMeta(, 7eea9210-6569-463f-b994-5c291588057f, 40, Finished, Available, Finished, False)

BSE MCAP Found: 4594
Final Market Cap Found: 4700


In [12]:
sector_df = sector_df[
    [
        "symbol",
        "sector",
        "industry"
    ]
].copy()

sector_df = sector_df.drop_duplicates(
    subset=["symbol"]
)

print(
    "Lookup Rows:",
    len(sector_df)
)

StatementMeta(, 7eea9210-6569-463f-b994-5c291588057f, 41, Finished, Available, Finished, False)

Lookup Rows: 2347


In [13]:
sec_master = sec_master.merge(
    sector_df,
    on="symbol",
    how="left"
)

print(
    "Sector Found:",
    sec_master["sector"].notna().sum()
)

print(
    "Industry Found:",
    sec_master["industry"].notna().sum()
)

StatementMeta(, 7eea9210-6569-463f-b994-5c291588057f, 42, Finished, Available, Finished, False)

Sector Found: 1491
Industry Found: 1491


In [14]:
security_master = sec_master[
    [
        "isin",

        "symbol",
        "company_name",

        "bse_security_code",

        "cmp",
        "market_cap",

        "sector",
        "industry",

        "face_value"
    ]
].copy()

print(
    "Rows:",
    len(security_master)
)

print(
    "Columns:",
    len(security_master.columns)
)

security_master.head()

StatementMeta(, 7eea9210-6569-463f-b994-5c291588057f, 45, Finished, Available, Finished, False)

Rows: 4963
Columns: 9


,isin,symbol,company_name,bse_security_code,cmp,market_cap,sector,industry,face_value
0,-,ADSDIAG,A.D.S. Diagnostics Ltd.,523031.0,NaN,NaN,NaN,NaN,10.0
1,IN9075K01029,ANNVRPP,ANNVRRIDHHI VENTURES LIMITED,890229.0,NaN,27.90,NaN,NaN,10.0
2,IN90GGO01013,KRISHPP,KRISHIVAL FOODS LIMITED,890232.0,NaN,42.70,NaN,NaN,10.0
3,IN90NN701011,ATLPP,Allcargo Terminals Limited,890228.0,NaN,38.61,NaN,NaN,2.0
4,IN9161F01019,ANIRITPP,ANIRIT VENTURES LIMITED,890231.0,NaN,35.70,NaN,NaN,10.0


In [15]:
security_master_spark = spark.createDataFrame(
    security_master
)

security_master_spark.write.mode(
    "overwrite"
).format(
    "delta"
).saveAsTable(
    "sec_master"
)

print("SUCCESS")

print(
    "Rows:",
    security_master_spark.count()
)

StatementMeta(, 7eea9210-6569-463f-b994-5c291588057f, 47, Finished, Available, Finished, False)

SUCCESS
Rows: 4963


In [1]:
# print("trade_daily:", spark.table("trade_daily").count())
# print("bulk_daily:", spark.table("bulk_daily").count())
# print("block_daily:", spark.table("block_daily").count())
# print("sec_master:", spark.table("sec_master").count())

StatementMeta(, 1dbe238f-ec96-4513-9fc6-29574488e8d1, 3, Finished, Available, Finished, False)

trade_daily: 14580830
bulk_daily: 513780
block_daily: 14471
sec_master: 4963
